# BhashaVerse/OneNMT adaptation — one experiment settings cell

Edit **only** the next cell per fresh session; execute in order. Shared parallel-data conversion and seeded splitting; model-specific **FairseqCodec/SentencePiece** route validation. For this checkpoint, independent upstream generation parity is required before accepting research outcomes.

Development loss and best-checkpoint selection are always available in the trainer. Baselines, test inference and publication reporting are optional. See `docs/experiment-guide.md` and `docs/notebook-platforms.md`.


## 1. Experiment settings — edit only here


In [ ]:
# EDIT ONLY THIS CELL FOR A BHASHAVERSE / ONEMT EXPERIMENT.
EXPERIMENT = {
    "data_mode": "aligned_text",   # "aligned_text", "tsv", "jsonl", "pre_split", "example".
    "source_lang": "mai_Deva",     # Card-listed example; verify others independently.
    "target_lang": "hin_Deva",
    "source_text": "/kaggle/input/YOUR_DATASET/mai.txt",
    "target_text": "/kaggle/input/YOUR_DATASET/hin.txt",
    "tsv_file": "/kaggle/input/YOUR_DATASET/parallel.tsv",
    "tsv_has_header": False,
    "parallel_jsonl": "data/local/mai-hin/parallel.jsonl",
    "split_directory": "data/local/mai-hin/splits",
    "dataset_id": "corpus-v1",
    "split_seed": 42, "split_ratios": (0.80, 0.10, 0.10),
    "group_by_document": True,
    "validate_data": False,        # True for complete audit.
    "data_report": False,
    "model_assets": "data/local/onenmt-original", # Verified local HF MBart-format assets.
    "conditioning": "text_prefix", # BhashaVerse direction prefix, not NLLB tags.
    "finetune_mode": "lora",        # New embedding rows -> full training.
    "train_seed": 42, "epochs": 3, "max_steps": -1,
    "batch_size": 1, "gradient_accumulation_steps": 16,
    "learning_rate": 2e-4, "max_length": 128,
    "max_new_tokens": 64, "beams": 4,
    "precision": "fp16", "gradient_checkpointing": True, "lora_rank": 8,
    "run_label": "pilot-1",
    "check_assets": True,          # Check dictionary/SPM before training; recommended.
    "allow_training": False,       # Enable only after upstream checkpoint parity check.
    "pretrain_baseline": False,    # Extra untuned model dev inference.
    "baseline_dev_size": 100,
    "posttrain_test": False, "publication_report": False,
    "test_baseline": False,
    "save_archive": True, "include_checkpoints": False,
    "setup_mode": "train,repairs",  # GPU training + repair utilities.
    "refresh_repo": False,
}


## 2. Environment and repository

Select GPU. Use a reviewed local compatible model checkpoint.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = (Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir() else Path.cwd().resolve())
os.environ["USE_TF"], os.environ["USE_FLAX"] = "0", "0"
if not (ROOT / "pyproject.toml").is_file():
    if ROOT.exists(): raise ValueError(f"Checkout exists but is incomplete: {ROOT}")
    subprocess.run(["git", "clone", "https://github.com/bhaashik/BhaashikMTFinetuning.git",
                    str(ROOT)], check=True)
elif EXPERIMENT["refresh_repo"]:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if any(m in sys.modules for m in ("torch","transformers","peft","datasets")):
    raise RuntimeError("Restart session before installing model packages.")
cmd = [sys.executable, str(ROOT / "scripts" / "setup_notebook.py"),
       "--extras", EXPERIMENT["setup_mode"]]
if EXPERIMENT["setup_mode"].startswith("train"):
    cmd.append("--require-cuda")
subprocess.run(cmd, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="bhashaverse")
print("Checkout:", ROOT)
# Use an independently reviewed HF MBart-compatible BhashaVerse checkpoint.
# Do not substitute generic AutoTokenizer; the Fairseq dictionary IDs matter.


## 3. Prepare and split parallel data

One input format, one split seed, shared for cross-model comparison.


In [ ]:
from bhaashik.experiment_workflow import (
    prepare_parallel_data, experiment_paths, training_parameters,
    write_experiment_config, resolve_path, report_data_if_enabled,
)
PREPARED = prepare_parallel_data(ROOT, EXPERIMENT, model_family="bhashaverse")
PATHS = experiment_paths(ROOT, EXPERIMENT, family="bhashaverse", prepared=PREPARED)
DATA, RUN, REPORT_DIR = PREPARED["data_dir"], PATHS["run"], PATHS["report"]
ASSETS = resolve_path(ROOT, EXPERIMENT["model_assets"])
report_data_if_enabled(EXPERIMENT, PREPARED, REPORT_DIR)
TRAIN_CONFIG = training_parameters(EXPERIMENT, PREPARED, PATHS,
                                   model_family="bhashaverse")
TRAIN_CONFIG_PATH = write_experiment_config(EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG)
print("Model assets:", ASSETS, "\nRun output:", RUN)


## 4. Check tokenizer/dictionary/checkpoint

Required before full research training; does not alter the original assets.


In [ ]:
if EXPERIMENT["check_assets"]:
    from bhaashik.preflight import FairseqCodec, checkpoint_layout
    from bhaashik.bhashaverse_adaptation import validate_prefix_routes
    from bhaashik.data import read_jsonl
    codec = FairseqCodec(ASSETS)
    layout = checkpoint_layout(codec)
    assert layout["weight_layout_checked"] and not layout["issues"], layout["issues"]
    rows = read_jsonl(DATA / "train.jsonl") + read_jsonl(DATA / "dev.jsonl")
    print("Representable direction prefixes:", validate_prefix_routes(codec, rows))
else:
    print("Asset compatibility checks disabled (training should not be published unverified).")


## 5. Optional unfinetuned model development baseline

Off by default; unrelated to development LOSS during training.


In [ ]:
if EXPERIMENT["pretrain_baseline"]:
    from bhaashik.model_comparisons import select_dev_sample
    from bhaashik.bhashaverse_adaptation import generate_bhashaverse_dev
    from bhaashik.pretrain_review import compare_dev
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    sample = REPORT_DIR / "dev_sample.jsonl"
    if not sample.exists():
        select_dev_sample(DATA / "dev.jsonl", sample,
                          sample_size=EXPERIMENT["baseline_dev_size"],
                          seed=EXPERIMENT["split_seed"])
    original = REPORT_DIR / "bhashaverse-original-dev.jsonl"
    if not original.exists():
        generate_bhashaverse_dev(
            ASSETS, sample, original, batch_size=EXPERIMENT["batch_size"],
            max_new_tokens=EXPERIMENT["max_new_tokens"])
    print(compare_dev(sample, {"BhashaVerse original": original}, REPORT_DIR))
else:
    print("Pre-training generation OFF (development loss is still used during training).")


## 6. Finetune

Off until the model assets and upstream parity are independently validated.


In [ ]:
if EXPERIMENT["allow_training"]:
    assert EXPERIMENT["setup_mode"].startswith("train")
    assert EXPERIMENT["check_assets"], (
        "Enable and pass checkpoint/dictionary preflight before training.")
    from bhaashik.bhashaverse_adaptation import train_bhashaverse
    # Dev loss is computed each epoch and lowest-loss checkpoint selected.
    print("Development loss:", train_bhashaverse(TRAIN_CONFIG_PATH))
    print("Exported model:", RUN)
else:
    print("BhashaVerse training OFF pending checkpoint parity and assets.")


## 7. Optional held-out test and paper metrics

Never use test results for hyperparameter selection.


In [ ]:
if EXPERIMENT["posttrain_test"]:
    assert (RUN / "run_manifest.json").is_file(), "Train or attach a model first."
    from bhaashik.bhashaverse_adaptation import generate_bhashaverse_dev
    predictions = RUN / "predictions.jsonl"
    if not predictions.exists():
        generate_bhashaverse_dev(
            RUN, DATA / "test.jsonl", predictions,
            batch_size=EXPERIMENT["batch_size"],
            max_new_tokens=EXPERIMENT["max_new_tokens"])
    print("Held-out predictions:", predictions)
    if EXPERIMENT["publication_report"]:
        from bhaashik.evaluation import evaluate, export_errors
        print(evaluate(predictions, RUN / "evaluation.json")["scores"])
        export_errors(predictions, RUN / "annotations.csv")
else:
    print("Final test and publication reports OFF.")


## 8. Archive the model

Retain original dictionary/SPM and optionally optimizer checkpoints.


In [ ]:
from bhaashik.experiment_workflow import save_optional_archive
if (RUN / "run_manifest.json").is_file():
    archive = save_optional_archive(
        ROOT, EXPERIMENT, PATHS,
        report=REPORT_DIR if REPORT_DIR.is_dir() else None)
    if archive: print("Save output ZIP:", archive)
else:
    print("No trained checkpoint yet; ZIP skipped.")
